# Module 2:  Vector Search
In this module, we implement vector search, which is a more robust form of search that matches documents based on meaning rather than just matching exact keywords in a query with keywords in documents.  

## What is vector space?  
The reason why it's call vector search is because the text is turned into vectors (a series of numbers representing the text).  This is also known as "embedding" the text in a vector space.  

### Word embeddings and sentence embeddings
A word's vector represents it at a point in a multi-dimensional space.  Words with similar meanings are closer together in this multidimensional space, while those whose meanings are dissimilar are further apart.  

Likewise for sentences:  an entire sentence can be embedded as a vector in multi-dimensional space as a single point, not just as a bunch of words in isolation.  This does a better job of representing the meaning of that sentence than the "bag of words" approach used in simpler search methodologies.  

In the case of our FAQ, each question and answer (each "document") is embedded in vector space.  When we ask a question, that question is embedded as a vector into that same multi-dimensional space.  The model finds the documents closest in space to the question (its nearest neighbors) and produces these as the search results.  

## The vector search process 
_(The text below is verbatim from the lesson because it lays out what we're going to be doing in each lesson section.  If the text in a section is not designated as verbatim from the lesson section, it's my own commentary and learning summary.)_

We run vector search in two stages.

* Offline (indexing): we convert all documents into vectors (arrays of numbers) and store them in an index.
* Online (querying): we convert the user's query into a vector with the same model, then find the closest document vectors by similarity.

An embedding model produces these vectors. It's a neural network trained to capture meaning, so texts that mean similar things land on similar vectors. We measure how close two vectors are with a distance metric. The most common one is cosine similarity.

Cosine similarity measures the angle between two vectors:

* Vectors pointing in the same direction: similarity close to 1 (similar)
* Vectors at right angles: similarity close to 0 (unrelated)
* Vectors pointing in opposite directions: similarity close to -1 (opposite meaning)

The larger the cosine similarity, the more similar the two texts are in meaning.

### Building vector search
We'll take the same FAQ dataset from module 1 and build vector search with three tools:

1. ```minsearch``` - in-memory vector search (simplest, good for experiments)
2. ```sqlitesearch``` - persistent vector search backed by SQLite (production-friendly, same API as minsearch)
3. ```PGVector``` - vector search in PostgreSQL (scalable, runs in Docker)

Then we'll plug vector search into our RAG pipeline.

### Introduction to Vector Search using SBERT

To begin, we're going to be using Sentence Transformers (SBERT) a Python module for using and training state-of-the-art embedding and reranker models.  Sentence Transformers was created by UKP Lab and is being maintained by 🤗 Hugging Face.  

I've installed SBERT in my environment, so I can start using this model in the lesson.  

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2') 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Here are a couple questions we'll use in this process:

In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [3]:
v1 = model.encode(q1)

We've produced a vector by encoding the question in the model.  

Let's see the shape of the vector:

In [4]:
v1.shape

(384,)

The vector has 384 values.

Let's encode the second question:

In [5]:
v2 = model.encode(q2)

In [6]:
v2.shape

(384,)

New question, new encoding:

In [7]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

As with the questions, we'll encode the document (answer) as a vector:

In [8]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

To find out the similarity between the question and the document (answer), we perform vector multiplication.  The higher the score, the better the match.  A score close to zero indicate that the model has found little or no similarity between the two vectors, whereas a score closer to 1 indicates a very strong similarity between the two vectors.

In [9]:
v1.dot(dv)

np.float32(0.32332397)

This score is pretty good, but not great. This makes sense:  the question asks about whether a student can join the course after the start date; the answer, while related to the concept of joining the course, does not specifically address the question of joining the course after the start date.  

Let's try another question:

In [10]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [11]:
v2.dot(dv)

np.float32(0.019730523)

Here the score is much lower.  If we look at the original value for ```d``` ("You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."), we can see that ```d``` does not contain content related to installing Docker on Windows.  

## Embedding Our Dataset

The simple example above shows a trivial case where a question is compared to a single answer for similarity.  For our search engine to be useful, we need to be able to pull in a lot more information.  

In the previous module, we created and used an ingestion script to pull in the course FAQ information into our pipeline.  Let's pull the script now so that we can use it.

### Loading the data

In [12]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

### Generating embeddings

In [13]:
from ingest import load_faq_data

documents = load_faq_data()

Let's see what we have, taking look at a single item:

In [14]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

This is a python dictionary containing a single question and answer from the FAQ database.  We need to turn this python dictionary into the proper text that we can embed into vector space.  

In [15]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

Taking a look at one element:

In [16]:
texts[10]

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [17]:
# check the number of texts within texts

len(texts)

1350

We have to do this for every individual text (Q&A item) in this in the FAQ corpus, which contains 1,350 texts.  If we try to load them all at once, it will take a long time and we can't see what's happening inside.  Ingesting the texts in batches allows us to watch how it's going as the process proceeds. 

In [18]:
# Shows a progress bar so we can see how long it takes to encode all the texts

from tqdm.auto import tqdm

Let's chunk the dataset into batches of 50 and encode each batch:

In [19]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [20]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [21]:
import numpy as np
X = np.array(vectors)

In [22]:
X

array([[-0.02670618, -0.12245759,  0.01594416, ..., -0.00230645,
        -0.11218396, -0.02365561],
       [-0.01099554, -0.1107475 , -0.02536939, ...,  0.09022234,
        -0.02697358,  0.01975662],
       [-0.08896555, -0.06128182,  0.00775604, ...,  0.04059714,
         0.00479282, -0.02745941],
       ...,
       [-0.03652922,  0.01415427, -0.06838643, ...,  0.04316792,
         0.08105534, -0.02148628],
       [-0.13091592, -0.06990599, -0.00931885, ..., -0.00044336,
        -0.01285727,  0.01426919],
       [-0.07984785,  0.0192698 ,  0.0254498 , ..., -0.03368026,
        -0.01884023,  0.05837052]], shape=(1350, 384), dtype=float32)

In [23]:
scores = X.dot(v1)

In [24]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

In [25]:

documents[553]

{'course': 'llm-zoomcamp',
 'section': 'Module 1: RAG',
 'question': 'OpenAI: Error when running OpenAI responses.create command',
 'answer': 'You may receive the following error when running the OpenAI `responses.create` command due to insufficient credits in your OpenAI account:\n\n```\nOpenAI API Error: Insufficient credits\n```',
 'doc_id': 'f5df151c59'}

In [26]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [27]:

scores[top5]

array([0.7629411 , 0.7579371 , 0.71921337, 0.6536313 , 0.5601001 ],
      dtype=float32)

In [28]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.71921337
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'

In [29]:
top5

array([  2, 625, 907, 538,   7])

In [30]:
top5 = np.argsort(-scores)[:5]

In [31]:
top5

array([  2, 625, 907, 538,   7])

In [32]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [33]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

## RAG with Vector Search
_(The text below is from the lesson, with minor edits.)_

In module 1, we built a RAG pipeline with three steps:

    def rag(question):
        search_results = search(question)
        user_prompt = build_prompt(question, search_results)
        return llm(user_prompt)

The search step used keyword search. Now we swap in vector search. Because RAG is modular, search is the only step we touch. The build prompt and the LLM call stay exactly as before.

In module 1 we put all the RAG logic into a ```RAGBase``` helper class. It has ```search```, ```build_prompt```, and ```llm``` methods, so we only need to override ```search```.

Download ```rag_helper.py``` (and ```ingest.py``` if you didn't get it earlier) into your project:

In [35]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [36]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [37]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [38]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [39]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [40]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [41]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [42]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

## Vector Search with sqlitesearch 
_(The text below draws on the lesson text but includes my notes from the video lessons as well.)_

In the previous section we used ```minsearch``` for vector search.

It works, but it has three problems:

1. It rebuilds the index on every startup
2. It keeps everything in memory
3. It searches by brute force

**Rebuilding the index every time**--With text search, indexing was fast because we didn't embed anything. With vector search, indexing runs a neural network over every document, so it takes a bit of time our dataset. 

**Keeping everything in memory**--Keeping everything in memory is fine here because our dataset is relative small, but a larger dataset would require too much space.

**Brute-force search**--For every query we compare the query vector against every single document. With 1,000 documents, this is fine; in fact, it's probably faster than anything 'smarter'. But as the dataset grows past 10,000 or so, the process slows down.  This is *exact nearest neighbor (NN) search*. We score the query against every document and pick the top ones. It always finds the true top matches, but it pays for that by touching everything.  

*Approximate nearest neighbor (ANN) search* takes a shortcut. Instead of comparing against everything, it first narrows down to a region of likely matches. Then it scores only within that region. It may miss the absolute best match, but the results are still good and it's much faster.  That's what we will implement below.  

### What is sqlitesearch and why are we using it?

We'll be using ```sqlitesearch```, a vector search library Alexey created.  It has sqlite under the hood, so it's a proper database.  Also, everything is persisted--meaning that you can put vectors in one process and load vectors from another process.    We split the process into two parts:  ingestion and deployment.  The FAQ index is created once in the ingestion process and is ready to go every time we start up another search. 

In [43]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [45]:
vs_index.clear()

In [46]:
vs_index.fit(vectors, documents)

In [47]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [48]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [51]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [50]:
vs_index.close()